# Home energy use

Did the time-of-use tariff move electricity use out of the evening peak, and did it lower the bills? The data is synthetic: 360 homes over 2025, made for the Whybook demo.

## Load

In [1]:
import whybook
import pandas as pd
import statsmodels.formula.api as smf
from energy import BASE_TEMP_C, MIN_COVERAGE, add_degree_days, drop_gappy

In [2]:
homes = pd.read_csv("homes.csv", parse_dates=["tou_start"])
weather = pd.read_csv("weather.csv", parse_dates=["date"])
tariffs = pd.read_csv("tariffs.csv")
readings = pd.read_parquet("readings.parquet")
readings.head()

,home_id,date,kwh_import,kwh_peak,kwh_night,kwh_export
0,H001,2025-01-01,20.108999,4.708,3.310,0.0
1,H001,2025-01-02,19.910000,4.416,3.103,0.0
2,H001,2025-01-03,17.916000,4.653,3.066,0.0
3,H001,2025-01-04,22.320000,4.583,3.487,0.0
4,H001,2025-01-05,21.995001,4.982,3.313,0.0


In [3]:
impossible = readings[(readings["kwh_import"] <= 0) | (readings["kwh_import"] > 500)]
print(f"{len(impossible)} impossible readings from {impossible['home_id'].nunique()} homes")
impossible

9 impossible readings from 9 homes


,home_id,date,kwh_import,kwh_peak,kwh_night,kwh_export
26058,H073,2025-10-22,0.000000,0.000,0.000,3.394
27685,H078,2025-04-20,999.900024,2.284,1.884,0.000
39783,H111,2025-10-18,0.000000,0.000,0.000,0.000
44192,H124,2025-01-25,999.900024,3.570,23.218,0.000
62457,H175,2025-03-24,0.000000,0.000,0.000,6.690
63206,H177,2025-06-18,0.000000,0.000,0.000,0.000
89995,H251,2025-08-31,999.900024,1.735,1.624,9.220
113676,H317,2025-10-27,0.000000,0.000,0.000,0.000
120897,H338,2025-01-01,999.900024,1.862,1.576,0.000


In [4]:
daily = (
    readings.drop(impossible.index)
    .pipe(drop_gappy)
    .assign(date=lambda d: pd.to_datetime(d["date"]))
    .merge(homes, on="home_id")
    .merge(weather, on=["region", "date"])
    .pipe(add_degree_days)
    .assign(
        month=lambda d: d["date"].dt.month,
        peak_share=lambda d: d["kwh_peak"] / d["kwh_import"],
        tou_active=lambda d: d["date"] >= d["tou_start"],
    )
)
daily.head()

,home_id,date,kwh_import,kwh_peak,kwh_night,kwh_export,region,floor_area_m2,occupants,built,...,tariff,tou_start,mean_temp_c,min_temp_c,sunshine_hours,rain_mm,hdd,month,peak_share,tou_active
0,H001,2025-01-01,20.108999,4.708,3.310,0.0,coast,132,5,before 1950,...,flat,NaT,5.7,2.7,1.0,4.7,9.8,1,0.234124,False
1,H001,2025-01-02,19.910000,4.416,3.103,0.0,coast,132,5,before 1950,...,flat,NaT,5.2,0.2,1.9,2.3,10.3,1,0.221798,False
2,H001,2025-01-03,17.916000,4.653,3.066,0.0,coast,132,5,before 1950,...,flat,NaT,5.1,-0.5,1.1,0.0,10.4,1,0.259712,False
3,H001,2025-01-04,22.320000,4.583,3.487,0.0,coast,132,5,before 1950,...,flat,NaT,5.6,2.6,0.0,0.0,9.9,1,0.205332,False
4,H001,2025-01-05,21.995001,4.982,3.313,0.0,coast,132,5,before 1950,...,flat,NaT,5.2,0.5,3.4,0.0,10.3,1,0.226506,False


## Use

In [5]:
monthly = daily.groupby(["home_id", "heating", "month"], as_index=False, observed=True).agg(
    kwh_import=("kwh_import", "mean"), hdd=("hdd", "mean")
)
whybook.ribbon(monthly, x="month", y="kwh_import", by="heating", title="Daily use by month, by heating")

<whybook ribbon: Daily use by month, by heating>

In [6]:
per_home = (
    daily.groupby(["home_id", "heating"], as_index=False, observed=True)["kwh_import"]
    .mean()
    .rename(columns={"kwh_import": "kwh_per_day"})
)
whybook.bars(per_home, x="heating", y="kwh_per_day", title="Mean daily use per home, by heating")
per_home.groupby("heating", observed=True)["kwh_per_day"].describe().round(1)

<whybook bars: Mean daily use per home, by heating>

,count,mean,std,min,25%,50%,75%,max
heating,,,,,,,,
electric storage,61.0,35.9,10.3,19.3,29.7,34.4,38.8,75.1
gas boiler,199.0,11.9,4.1,4.5,8.5,11.1,14.8,22.8
heat pump,79.0,20.2,6.1,7.9,16.3,20.2,24.4,37.3


## Heating

In [7]:
model_data = monthly.merge(homes[["home_id", "occupants", "floor_area_m2"]], on="home_id")
heating_fit = smf.mixedlm(
    "kwh_import ~ hdd * heating + occupants + floor_area_m2", data=model_data, groups="home_id"
).fit()
heating_fit.summary().tables[1]

,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
Intercept,1.652,1.108,1.492,0.136,-0.518,3.823
heating[T.gas boiler],-1.816,0.720,-2.521,0.012,-3.227,-0.404
heating[T.heat pump],-2.093,0.840,-2.492,0.013,-3.740,-0.447
hdd,4.045,0.032,126.721,0.000,3.983,4.108
hdd:heating[T.gas boiler],-3.921,0.036,-107.964,0.000,-3.992,-3.849
hdd:heating[T.heat pump],-2.552,0.042,-60.746,0.000,-2.635,-2.470
occupants,2.332,0.203,11.499,0.000,1.934,2.729
floor_area_m2,0.062,0.008,7.462,0.000,0.045,0.078
home_id Var,20.873,0.457,,,,


In [8]:
sweep = []
for i, base in enumerate([12.0, 14.0, 15.5, 18.0]):
    whybook.progress(i / 4, f"base {base} °C")
    swept = (
        daily.pipe(add_degree_days, base=base)
        .groupby(["home_id", "heating", "month"], as_index=False, observed=True)
        .agg(kwh_import=("kwh_import", "mean"), hdd=("hdd", "mean"))
        .merge(homes[["home_id", "occupants", "floor_area_m2"]], on="home_id")
    )
    fit = smf.mixedlm("kwh_import ~ hdd * heating + occupants + floor_area_m2", data=swept, groups="home_id").fit(reml=False)
    sweep.append(
        {
            "base_temp_c": base,
            "heat_pump_kwh_per_degree": fit.params["hdd"] + fit.params["hdd:heating[T.heat pump]"],
            "log_likelihood": fit.llf,
        }
    )
whybook.progress(1.0, "done")
base_sweep = pd.DataFrame(sweep).round(3)
base_sweep

[100%] done

,base_temp_c,heat_pump_kwh_per_degree,log_likelihood
0,12.0,1.977,-11824.522
1,14.0,1.659,-11722.652
2,15.5,1.493,-11806.695
3,18.0,1.327,-12023.224


## The time-of-use trial

In [9]:
who_switched = (
    homes.groupby("tariff")[["has_ev", "has_solar"]].mean().round(2).assign(homes=homes["tariff"].value_counts())
)
who_switched

,has_ev,has_solar,homes
tariff,,,
flat,0.13,0.20,267
time of use,0.58,0.26,93


In [10]:
peak_monthly = daily.groupby(["home_id", "tariff", "month"], as_index=False, observed=True)["peak_share"].mean()
whybook.ribbon(peak_monthly, x="month", y="peak_share", by="tariff", title="Share of use between 16:00 and 19:00, by tariff")

<whybook ribbon: Share of use between 16:00 and 19:00, by tariff>

In [11]:
after = daily[daily["month"] >= 10]
naive = after.groupby("tariff")["peak_share"].mean().round(3)
print(f"October to December: {naive['flat']:.1%} of use falls between 16:00 and 19:00 on the flat tariff, {naive['time of use']:.1%} on time of use")
naive

October to December: 21.2% of use falls between 16:00 and 19:00 on the flat tariff, 17.9% on time of use


tariff
flat           0.212
time of use    0.179
Name: peak_share, dtype: float32

In [12]:
switch_data = peak_monthly.merge(homes[["home_id", "tou_start"]], on="home_id").assign(
    tou_active=lambda d: d["tou_start"].dt.month < d["month"]
)
did_fit = smf.mixedlm("peak_share ~ tou_active + C(month)", data=switch_data, groups="home_id").fit()
did_fit.summary().tables[1].loc[["tou_active[T.True]"]]

/tmp/ipykernel_151142/2283361393.py:4: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  did_fit = smf.mixedlm("peak_share ~ tou_active + C(month)", data=switch_data, groups="home_id").fit()


,Coef.,Std.Err.,z,P>|z|,[0.025,0.975]
tou_active[T.True],-0.053,0.003,-18.994,0.000,-0.059,-0.048


In [13]:
half = pd.read_parquet("half_hourly.parquet").merge(homes[["home_id", "tariff"]], on="home_id")
half = half.assign(month=half["timestamp"].dt.month, hour=half["timestamp"].dt.hour + half["timestamp"].dt.minute / 60)
day_shape = half.groupby(["home_id", "tariff", "month", "hour"], as_index=False, observed=True)["kwh"].mean()
february = day_shape[day_shape["month"] == 2]
october = day_shape[day_shape["month"] == 10]
whybook.ribbon(february, x="hour", y="kwh", by="tariff", title="Use by half hour in a February week, by tariff")
whybook.ribbon(october, x="hour", y="kwh", by="tariff", title="Use by half hour in an October week, by tariff")

<whybook ribbon: Use by half hour in a February week, by tariff>

<whybook ribbon: Use by half hour in an October week, by tariff>

## Bills

In [14]:
flat_price = tariffs.loc[tariffs["tariff"] == "flat", "eur_per_kwh"].iloc[0]
tou_price = tariffs[tariffs["tariff"] == "time of use"].set_index("period")["eur_per_kwh"]
standing = tariffs.groupby("tariff")["standing_eur_per_day"].first()
flat_cost = daily["kwh_import"] * flat_price + standing["flat"]
tou_cost = (
    daily["kwh_night"] * tou_price["night"]
    + daily["kwh_peak"] * tou_price["peak"]
    + (daily["kwh_import"] - daily["kwh_night"] - daily["kwh_peak"]) * tou_price["day"]
    + standing["time of use"]
)
billed = daily.assign(cost_eur=flat_cost.where(~daily["tou_active"], tou_cost), flat_eur=flat_cost)
bills = billed.groupby(["home_id", "tariff", "month"], as_index=False, observed=True)[["cost_eur", "flat_eur"]].sum()
late_bills = bills[bills["month"] >= 10]
whybook.bars(late_bills, x="tariff", y="cost_eur", title="Monthly bill from October to December, by tariff")
switched = billed[billed["tou_active"]]
monthly_saving = (switched["flat_eur"] - switched["cost_eur"]).sum() / switched.groupby(["home_id", "month"]).ngroups
print(f"On the new tariff, a switcher pays {monthly_saving:.2f} EUR a month less than the flat tariff would charge for the same use")
bills.pivot_table(index="month", columns="tariff", values="cost_eur", aggfunc="mean").round(2)

<whybook bars: Monthly bill from October to December, by tariff>

On the new tariff, a switcher pays 24.42 EUR a month less than the flat tariff would charge for the same use


tariff,flat,time of use
month,,
1,253.699997,256.890015
2,225.679993,230.759995
3,223.990005,222.179993
4,168.110001,165.399994
5,139.899994,136.610001
6,124.220001,120.550003
7,124.139999,121.879997
8,126.480003,123.900002
9,134.929993,128.750000


## Summary

Heat pump homes use 1.5 kWh more a day for each degree of cold below 15.5 °C, and homes with storage heaters 4.0 kWh. After the switch, the share of use between 16:00 and 19:00 fell by 5.3 points in the homes on the time-of-use tariff, against the flat homes over the same months. A plain comparison after the switch says 3.3 points, because the homes that switched used more in the evening to begin with: 58% of them own an electric car, against 13% of the others. On the new tariff a switcher pays 24.42 EUR a month less than the flat tariff would charge for the same use. 9 impossible readings are left out, and so are 21 homes with long gaps in their readings.